# Industrial Predictive Maintenance with Machine Learning

**Objective:** Develop a machine learning system to predict equipment failures in industrial processes using historical sensor data (temperature, pressure, vibration).

**Dataset:** [AI4I 2020 Predictive Maintenance Dataset](https://www.kaggle.com/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020) — 10,000 data points with 14 features simulating real industrial conditions.

**Models compared:** Decision Tree, Support Vector Machine (SVM), Neural Network (MLP)

**Tools:** Python, scikit-learn, pandas, matplotlib, seaborn

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, f1_score, recall_score,
                             precision_score, accuracy_score)

import warnings
warnings.filterwarnings('ignore')

## 2. Data Loading & Exploration

In [ ]:
# Load the AI4I 2020 Predictive Maintenance dataset
df = pd.read_csv('ai4i2020.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# General information about the dataset
print("=== DATASET INFO ===")
print(df.info())

print("\n=== BASIC STATISTICS ===")
df.describe()

In [ ]:
# Check class distribution (failures vs no failures)
print("=== FAILURE DISTRIBUTION ===")
print(df['Machine failure'].value_counts())
print(f"\nFailure rate: {df['Machine failure'].mean()*100:.2f}%")

# Visualize class distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='Machine failure', data=df, palette='Set2')
plt.title('Failure Distribution')
plt.xlabel('0 = No Failure | 1 = Failure')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap between numerical features
plt.figure(figsize=(10, 6))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

**Key decisions:**
- Remove `UDI` and `Product ID` (identifiers, not features)
- Remove `TWF`, `HDF`, `PWF`, `OSF`, `RNF` (failure subtypes that cause data leakage)
- Encode `Type` column (L=0, M=1, H=2)
- Use stratified train/test split to preserve failure ratio
- Scale features with StandardScaler (critical for SVM and Neural Network)

In [ ]:
# Remove irrelevant columns and failure subtypes (prevent data leakage)
df_clean = df.drop(['UDI', 'Product ID', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'], axis=1)

# Encode product quality type: L (Low) = 0, M (Medium) = 1, H (High) = 2
df_clean['Type'] = df_clean['Type'].map({'L': 0, 'M': 1, 'H': 2})

print(f"Final columns: {df_clean.columns.tolist()}")
print(f"Shape: {df_clean.shape}")

In [ ]:
# Separate features (X) and target (y)
X = df_clean.drop('Machine failure', axis=1)
y = df_clean['Machine failure']

# Train/test split (80/20) with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape} | Test set: {X_test.shape}")
print(f"Failures in train: {y_train.sum()} | Failures in test: {y_test.sum()}")

# Feature scaling (important for SVM and Neural Network)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nPreprocessing completed.")

## 4. Model Training & Evaluation

Three classifiers are compared:
1. **Decision Tree** — interpretable, handles imbalanced data with `class_weight='balanced'`
2. **SVM (Support Vector Machine)** — effective in high-dimensional spaces
3. **Neural Network (MLP)** — two hidden layers (64, 32 neurons), captures non-linear patterns

In [ ]:
# Define models
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'SVM':           SVC(random_state=42, class_weight='balanced'),
    'Neural Network': MLPClassifier(random_state=42, max_iter=500, hidden_layer_sizes=(64, 32))
}

# Train, predict and evaluate each model
results = {}

for name, model in models.items():
    # Train
    model.fit(X_train_scaled, y_train)

    # Predict
    y_pred = model.predict(X_test_scaled)
    results[name] = y_pred

    # Classification report
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=['No Failure', 'Failure']))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Failure', 'Failure'])
    disp.plot(cmap='Blues')
    plt.title(f'Confusion Matrix - {name}')
    plt.tight_layout()
    plt.show()

## 5. Model Comparison

In [ ]:
# Collect metrics for all models
metrics = []
for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    metrics.append({
        'Model': name,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1-Score':  f1_score(y_test, y_pred)
    })

df_metrics = pd.DataFrame(metrics).set_index('Model')
print(df_metrics.round(3))

# Comparative bar chart
df_metrics.plot(kind='bar', figsize=(10, 5), ylim=(0, 1.1),
                colormap='Set2', edgecolor='black')
plt.title('Model Comparison - Failure Detection Metrics')
plt.ylabel('Score')
plt.xticks(rotation=15)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 6. Conclusions

**Key findings:**
- The dataset is highly imbalanced (~3.4% failure rate), making Recall the most critical metric
- `class_weight='balanced'` was applied to Decision Tree and SVM to handle imbalance
- All three models were evaluated on Accuracy, Precision, Recall and F1-Score

**Metric interpretation:**
- **Accuracy** — overall percentage of correct predictions
- **Precision** — of all predicted failures, how many were actual failures
- **Recall** — of all actual failures, how many were correctly detected (most important for maintenance)
- **F1-Score** — harmonic mean of Precision and Recall

**Next steps:**
- Hyperparameter tuning with GridSearchCV
- Feature importance analysis
- Deploy as interactive dashboard with Streamlit